# Análisis de noticias de Infobae con PLN

**Pregunta:** ¿qué se puede aprender de un corpus de noticias en español con técnicas de
procesamiento de lenguaje natural, y cómo se comparan esas técnicas entre sí?

**Datos:** notas publicadas en el feed RSS de Infobae, acumuladas con `src/actualizar_corpus.py`.
El feed solo entrega las últimas ~100 notas por consulta, así que el corpus se arma corriendo
el script en distintos momentos.

**Nota sobre los datos:** el corpus contiene el texto completo de las notas, que tiene derechos
de autor, por eso no se incluye en el repositorio. Para reconstruirlo, ejecutar el script.

In [73]:
# Las librerias pueden tardar un minuto en cargar.
# Librería estándar 
import time
from collections import Counter, defaultdict
from pathlib import Path

# Datos
import pandas as pd

# PLN
import spacy

#Librerias asociadas a ML
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#Librerias asociadas a BERTopic
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

from sentence_transformers import SentenceTransformer, util
SEED = 42    # semilla fija para que los resultados se puedan reproducir

In [74]:
RAIZ = Path.cwd().parent          # este notebook está en notebooks/
d = pd.read_csv(RAIZ / "data" / "noticias_infobae.csv")
n_antes = len(d)

d = d.dropna(subset=["texto"]).reset_index(drop=True)
print(f"Se descartaron {n_antes - len(d)} notas sin texto")

d["fecha"] = pd.to_datetime(d["fecha"], utc=True)
d["seccion"] = d["url"].str.split("/").str[3]
d["palabras"] = d["texto"].str.split().str.len()

print(d.shape)
print(d["fecha"].min(), "->", d["fecha"].max())
print(d["seccion"].value_counts())
print(d["palabras"].describe().round(0))

Se descartaron 3 notas sin texto
(2500, 7)
2026-10-23 19:00:54+00:00 -> 2026-10-27 03:27:26+00:00
seccion
america                    775
espana                     428
mexico                     277
colombia                   193
peru                       179
deportes                    62
tecno                       58
salud                       50
teleshow                    49
sociedad                    41
politica                    39
economia                    33
estados-unidos              26
noticias                    25
tendencias                  25
judiciales                  21
el-salvador                 19
panama                      19
entretenimiento             16
honduras                    16
cultura                     15
opinion                     15
malditos-nerds              14
cuba                        14
costa-rica                  13
guatemala                   12
nicaragua                   11
republica-dominicana         8
historias                 

## 2. Preprocesamiento

Se limpia el texto con spaCy: minúsculas, tokenización, eliminación de stopwords y
lematización. Se conservan las negaciones (`no`, `nunca`, `sin`...), porque spaCy las trata
como stopwords y descartarlas invierte el sentido de frases como "no me gustó".

Este texto limpio se usa con TF-IDF. Los modelos de embeddings y transformers reciben el
texto original, porque están entrenados con oraciones completas.

In [75]:
nlp = spacy.load("es_core_news_md", disable=["parser", "ner"])
negaciones = {"no", "nunca", "ni", "sin", "jamás", "tampoco"}


def limpiar(texto):
    doc = nlp(texto.lower())
    tokens = [t.lemma_.lower() for t in doc
              if (not t.is_stop or t.lower_ in negaciones) and t.is_alpha]
    return " ".join(tokens)


t0 = time.time()
d["limpio"] = d["texto"].apply(limpiar)


## 3. Predecir la edición del diario

**Tarea:** dado el texto de una nota, predecir si pertenece a la edición `america`, `espana`,
`mexico`, `peru` o `colombia` (la edición sale de la URL). Se usan solo estas cinco porque el
resto de las secciones tiene muy pocas notas.

**Aclaración:** la edición no es un tema. Una nota de deportes de México y una de política de
México tienen la misma etiqueta. La tarea mide si el vocabulario de una nota delata su origen.

**Evaluación:** validación cruzada estratificada de 5 folds, con F1 macro (promedia las clases por
igual) y comparada con un baseline que predice la misma edición (la más frecuente) para cada una de las notas.

### 3.1 Baseline: TF-IDF + regresión logística

In [76]:
EDICIONES = ["america", "espana", "mexico", "peru", "colombia"]
mask = d["seccion"].isin(EDICIONES)
d5 = d[mask]
y = d5["seccion"]

baseline = y.value_counts(normalize=True).max()
print(f"{len(d5)} notas | baseline (siempre la más frecuente): {baseline:.2f} de accuracy")

pipe_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=2, max_df=0.5)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=10)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pred_tfidf = cross_val_predict(pipe_tfidf, d5["limpio"], y, cv=cv)

print("F1 macro:", round(f1_score(y, pred_tfidf, average="macro"), 2))
print(classification_report(y, pred_tfidf))

etiq = sorted(EDICIONES)
print(pd.DataFrame(confusion_matrix(y, pred_tfidf, labels=etiq),
                   index=[f"real_{e}" for e in etiq],
                   columns=[f"pred_{e}" for e in etiq]))

1852 notas | baseline (siempre la más frecuente): 0.42 de accuracy
F1 macro: 0.85
              precision    recall  f1-score   support

     america       0.85      0.79      0.82       775
    colombia       0.92      0.91      0.91       193
      espana       0.71      0.79      0.75       428
      mexico       0.88      0.94      0.91       277
        peru       0.91      0.86      0.89       179

    accuracy                           0.83      1852
   macro avg       0.85      0.86      0.85      1852
weighted avg       0.83      0.83      0.83      1852

               pred_america  pred_colombia  pred_espana  pred_mexico  \
real_america            612             10          121           21   
real_colombia             5            175            5            7   
real_espana              82              0          339            5   
real_mexico              11              1            5          259   
real_peru                13              5            5            2 

### 3.2 Embeddings + regresión logística

Se representa cada nota con un embedding de 384 dimensiones
(`paraphrase-multilingual-MiniLM-L12-v2`), calculado sobre el texto original, y se entrena el
mismo clasificador con los mismos folds. El modelo lee solo los primeros ~128 tokens de cada nota.

In [77]:
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
E = modelo_emb.encode(d["texto"].tolist(), show_progress_bar=True)
print(E.shape)

E5 = E[mask.to_numpy()]
clf_emb = LogisticRegression(max_iter=2000, class_weight="balanced", C=10)
pred_emb = cross_val_predict(clf_emb, E5, y, cv=cv)

print("F1 macro TF-IDF:      ", round(f1_score(y, pred_tfidf, average="macro"), 2))
print("F1 macro embeddings:  ", round(f1_score(y, pred_emb, average="macro"), 2))
print(classification_report(y, pred_emb))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

(2500, 384)
F1 macro TF-IDF:       0.85
F1 macro embeddings:   0.65
              precision    recall  f1-score   support

     america       0.80      0.69      0.74       775
    colombia       0.55      0.65      0.60       193
      espana       0.66      0.69      0.68       428
      mexico       0.64      0.70      0.67       277
        peru       0.54      0.61      0.57       179

    accuracy                           0.68      1852
   macro avg       0.64      0.67      0.65      1852
weighted avg       0.69      0.68      0.68      1852



**Control:** TF-IDF lee la nota completa y el modelo de embeddings solo los primeros ~128 tokens.
Para comparar en igualdad de condiciones, se repite TF-IDF recortando cada nota a las primeras 90 palabras.

In [78]:
d["texto_corto"] = d["texto"].str.split().str[:90].str.join(" ")
d["limpio_corto"] = d["texto_corto"].apply(limpiar)

pred_tfidf_corto = cross_val_predict(pipe_tfidf, d.loc[mask, "limpio_corto"], y, cv=cv)

print("F1 macro TF-IDF (texto completo):    ", round(f1_score(y, pred_tfidf, average="macro"), 2))
print("F1 macro TF-IDF (primeras 90 pal.):  ", round(f1_score(y, pred_tfidf_corto, average="macro"), 2))
print("F1 macro embeddings:                 ", round(f1_score(y, pred_emb, average="macro"), 2))

F1 macro TF-IDF (texto completo):     0.85
F1 macro TF-IDF (primeras 90 pal.):   0.79
F1 macro embeddings:                  0.65


### 3.3 Conclusión

| Método | Texto que lee | F1 macro |
|---|---|---|
| Baseline (siempre `america`) | - | accuracy 0.42 |
| TF-IDF + regresión logística | nota completa | 0.85 |
| TF-IDF + regresión logística | primeras 90 palabras | 0.79 |
| Embeddings + regresión logística | primeros ~128 tokens | 0.65 |

- TF-IDF supera a los embeddings por 14 puntos con texto de largo comparable; otros 6 puntos
  de ventaja vienen de leer la nota completa.
- **Hipótesis (no verificada):** la edición se reconoce por palabras concretas (lugares, políticos,
  instituciones) que TF-IDF ve directamente, mientras que un embedding resume el tema.
- El error principal es la confusión entre `america` y `espana`: 203 de los 313 errores (65%). Se podría explorar si se debe a que `america` agrupa cables de agencia y temas generales.
  Se podría explorar si se debe a que `america` agrupa cables de agencia y temas generales.

**Limitaciones:** el corpus cubre unos 3 días, por lo que una misma noticia puede aparecer en
varias notas y quedar repartida entre entrenamiento y prueba, lo que infla el resultado.
Los hiperparámetros (`C=10`, `class_weight="balanced"`) se eligieron mirando resultados previos.

## 4. Descubrir temas sin etiquetas

Se agrupan las notas por similitud de significado, usando los embeddings, sin ninguna etiqueta.
Primero con KMeans (hay que elegir la cantidad de grupos) y luego con BERTopic (la elige solo).

### 4.1 KMeans: elegir la cantidad de grupos

El silhouette mide qué tan separados quedan los grupos (cerca de 1 es mejor). Con texto suele dar
valores bajos, así que sirve para comparar entre valores de `k`. La decisión final se toma leyendo
los grupos.

In [79]:
for k in [4, 6, 8, 10, 12, 15]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    etiquetas = km.fit_predict(E)
    sil = silhouette_score(E, etiquetas, metric="cosine")
    tamanos = sorted(pd.Series(etiquetas).value_counts().tolist(), reverse=True)
    print(k, round(sil, 3), tamanos)

4 0.067 [1013, 678, 446, 363]
6 0.094 [653, 621, 442, 346, 280, 158]
8 0.089 [549, 440, 356, 340, 305, 182, 180, 148]
10 0.1 [368, 346, 328, 327, 283, 255, 193, 151, 140, 109]
12 0.095 [345, 333, 278, 262, 252, 223, 188, 165, 138, 134, 102, 80]
15 0.094 [318, 233, 221, 217, 209, 193, 172, 163, 157, 139, 124, 121, 99, 72, 62]


### 4.2 Lectura de los grupos con k=10

Los silhouette con k mayor o igual a 6 dan resultados similares (un 10% de diferencia entre ellos). Se elige k=10 por estar en el medio de las otras pruebas, lo que da una tolerancia del valor de k hacia ambos lados del elegido. En este caso, la repartición de las noticias entre los clusters es la más equilibrada.

Para cada grupo se muestran las palabras que lo distinguen (TF-IDF calculado entre grupos) y los
3 títulos más cercanos a su centro, es decir, los más representativos.

In [80]:
def leer_clusters(k, n_titulos=3):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    etiquetas = km.fit_predict(E)
    dist = km.transform(E)

    docs = d.assign(c=etiquetas).groupby("c")["limpio"].apply(" ".join)
    tf = TfidfVectorizer(max_df=0.7)
    Xc = tf.fit_transform(docs)
    vocab = tf.get_feature_names_out()

    for i, c in enumerate(docs.index):
        idx = np.where(etiquetas == c)[0]
        centrales = idx[np.argsort(dist[idx, c])][:n_titulos]
        top = Xc[i].toarray()[0].argsort()[::-1][:8]
        print(f"\n=== Cluster {c} ({len(idx)} notas) ===")
        print("Palabras:", [vocab[j] for j in top])
        for j in centrales:
            print(" -", d["titulo"].iloc[j])
    return etiquetas

et10 = leer_clusters(10)
d["cluster_km"] = et10     


=== Cluster 0 (283 notas) ===
Palabras: ['película', 'komanche', 'conductora', 'musical', 'libro', 'festival', 'mamá', 'vestido']
 - De la Nahuala al Charro Negro: la saga “Las Leyendas” y su fórmula para dominar la taquilla nacional
 - El último mensaje de El Komanche se viraliza tras reportarse su muerte en México
 - Juls Janeiro evita mojarse en la guerra entre Beatriz Trapote y Belén Esteban pese a su apoyo público a su tía

=== Cluster 1 (368 notas) ===
Palabras: ['recomendar', 'clínico', 'proteína', 'salsa', 'grasa', 'queso', 'animal', 'humedad']
 - Aguas termales de Costa Rica podrían aliviar síntomas de osteoartritis y enfermedades de la piel, según especialista
 - Científicos argentinos diseñaron una proteína para tratar lesiones en la córnea tras una quemadura química
 - Cómo cuidar las hortensias cortadas para que duren más tiempo en el florero

=== Cluster 2 (327 notas) ===
Palabras: ['fiscalía', 'delito', 'policial', 'homicidio', 'crimen', 'juez', 'asesinato', 'alias']
 -

**Interpretación.** Leyendo los 3 títulos más cercanos al centro de cada grupo y las palabras que los distinguen, los 10 grupos corresponden a estos temas:

| Grupo | Notas | Tema |
|---|---|---|
| Cultura y espectáculos | 283 | cine, música, libros, festivales, farándula |
| Salud y vida cotidiana | 368 | salud, ciencia, recetas, jardinería |
| Crimen y justicia | 327 | delitos, homicidios, fiscalías, jueces |
| Lotería y sorteos | 109 | resultados de loterías y sorteos |
| Deportes | 328 | partidos y resultados |
| Elecciones en Brasil | 193 | balotaje, Lula, Bolsonaro y reacciones |
| Clima y fenómenos naturales | 151 | lluvias, temperaturas, sismos |
| Conflictos internacionales | 140 | Israel, Palestina, Ucrania, Irán |
| Economía | 255 | dólar, inflación, tasas, vivienda |
| Política | 346 | partidos, elecciones en España y Argentina, Congreso |

Observaciones:

- **El grupo de salud y vida cotidiana es el más heterogéneo:** reúne salud, ciencia, recetas y jardinería, y es el más grande del corpus.
- **Un solo hecho domina un grupo:** el de Elecciones en Brasil está formado casi por completo por notas sobre ese balotaje (resultado y reacciones). Es un tema puntual del período y no un tema permanente del diario. Además se superpone con Política: una nota sobre políticos argentinos que viajaron a Brasil quedó en este último.
- **El grupo de lotería parece incluir alguna nota deportiva**, porque entre sus palabras aparecen `balón` y `bayern`. No se verificó.

**Limitaciones:** la lectura se basó en pocos títulos por grupo, así que es una impresión y no una medición. `k=10` tiene el silhouette más alto (0.100), pero la diferencia con `k=12` (0.095) y `k=8` (0.089) es mínima, por lo que no hay un valor claramente mejor. La numeración de los grupos depende de la semilla y del corpus.

### 4.3 BERTopic

Se reutilizan los embeddings ya calculados (`E`) y el texto lematizado (`limpio`) para describir
cada tema con sus palabras características. Los parámetros principales son `min_cluster_size=30`
(mínimo de notas para formar un tema) y una semilla fija en UMAP para que el resultado sea
reproducible. Las notas que HDBSCAN no logra asignar a ningún tema quedan en el tema -1.

Se fijan `min_cluster_size=30` y `min_samples=3` explícitamente. Con `min_cluster_size=10` salían 82 temas, muchos de ellos un único hecho publicado en varias notas. Se exploraron combinaciones de ambos parámetros y se eligió esta por dar 24 temas de tamaño parejo, con un 11% de notas sin tema. Con `min_samples=5` el mismo `min_cluster_size` reunía 514 notas en un solo tema.

In [81]:
assert E.shape[0] == len(d)     # las filas de E y de d tienen que coincidir

topic_model = BERTopic(
    embedding_model=modelo_emb,
    umap_model=UMAP(n_neighbors=10, n_components=5, min_dist=0.0,
                    metric="cosine", random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=30, min_samples=3, prediction_data=True),
    vectorizer_model=CountVectorizer(min_df=2),
)
topics, probs = topic_model.fit_transform(d["limpio"].tolist(), embeddings=E)
d["topic"] = topics

info = topic_model.get_topic_info()
n_temas = (info["Topic"] != -1).sum()
n_out = (d["topic"] == -1).sum()
print(f"{n_temas} temas | {n_out} notas sin tema ({n_out / len(d):.0%})")
print(info[["Topic", "Count", "Name"]].head(20).to_string())

24 temas | 271 notas sin tema (11%)
    Topic  Count                                   Name
0      -1    271                     -1_no_año_él_dólar
1       0    270             0_partido_equipo_gol_punto
2       1    207                       1_no_él_año_vida
3       2    182                2_millón_no_empresa_año
4       3    182              3_no_gobierno_pp_elección
5       4    174      4_lula_bolsonaro_brasil_brasileño
6       5    169   5_no_investigación_presunto_judicial
7       6    147                   6_él_no_agua_persona
8       7    121           7_ataque_herido_ruso_militar
9       8     81      8_militar_dominicano_haití_fuerza
10      9     80                9_no_familia_policía_él
11     10     76                10_oro_balón_premio_año
12     11     60        11_sorteo_premio_ganador_número
13     12     59    12_temperatura_grado_clima_templado
14     13     58        13_aplicación_modelo_ia_usuario
15     14     42     14_colapinto_piloto_carrera_premio
16     15   

Los nombres de los temas incluyen palabras de relleno (`no`, `él`, `año`), porque el vectorizador
que los genera no descarta palabras frecuentes. Se recalculan con `max_df=0.5`, que elimina las
palabras presentes en más de la mitad de los temas. Esto solo cambia los nombres: la asignación
de notas a temas no se modifica.

In [82]:
vec_desc = CountVectorizer(min_df=2, max_df=0.5)
topic_model.update_topics(d["limpio"].tolist(), vectorizer_model=vec_desc)
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]].to_string())

    Topic  Count                                          Name
0      -1    271             -1_dólar_personaje_película_vucic
1       0    270                    0_gol_liga_torneo_atlético
2       1    207           1_cantante_película_conductora_cine
3       2    182   2_trimestre_financiamiento_inflación_billón
4       3    182                   3_pp_vox_senado_candidatura
5       4    174                 4_lula_bolsonaro_flávio_silva
6       5    169                 5_juez_prisión_sexual_acusado
7       6    147                 6_animal_proteína_grasa_queso
8       7    121                7_ucraniano_drón_saudí_ucrania
9       8     81       8_dominicano_haití_salvadoreño_pandilla
10      9     80  9_streamer_homicidio_fallecimiento_asesinato
11     10     76                10_balón_kane_messi_futbolista
12     11     60                 11_sorteo_boleto_super_podrás
13     12     59      12_templado_cálido_centígrado_termómetro
14     13     58                    13_ia_apple_youtube

### 4.4 Comparación entre KMeans y BERTopic

Se cruzan los 10 grupos de KMeans con los temas de BERTopic para ver si los dos métodos coinciden.
Cada fila es un tema de BERTopic y cada columna un grupo de KMeans; el número es la cantidad de
notas en esa combinación. Si un tema cae casi entero en una columna, los dos métodos están de
acuerdo.

Una aclaración para leer la tabla: los dos métodos usan los mismos embeddings, así que su
coincidencia es esperable y no es una validación independiente. El tema -1 reúne las notas que
BERTopic no pudo asignar, y su nombre no describe un tema.

In [83]:
# Numeración válida para SEED=42 y el corpus actual.
# Numeración válida para SEED=42 y corpus_congelado.csv; revisar si cambia alguno de los dos.
nombres_km = {
    0: "cultura",
    1: "salud/vida cotidiana",
    2: "crimen y justicia",
    3: "lotería",
    4: "deportes",
    5: "elecciones Brasil",
    6: "clima",
    7: "conflictos internac.",
    8: "economía",
    9: "política",
}
d["grupo_km"] = d["cluster_km"].map(nombres_km)

nombres_t = topic_model.get_topic_info().set_index("Topic")["Name"]
ct = pd.crosstab(d["topic"], d["grupo_km"])[list(nombres_km.values())]
ct.index = [f"{t}: {nombres_t[t][:28]}" for t in ct.index]
print(ct.to_string())

grupo_km                          cultura  salud/vida cotidiana  crimen y justicia  lotería  deportes  elecciones Brasil  clima  conflictos internac.  economía  política
-1: -1_dólar_personaje_película_       45                    44                 34        1         4                 37     11                    17        54        24
0: 0_gol_liga_torneo_atlético           2                     3                  1        8       250                  0      0                     1         0         5
1: 1_cantante_película_conducto       190                     7                  3        2         1                  0      3                     0         0         1
2: 2_trimestre_financiamiento_i         0                    31                  1        0         0                  0      1                     1       146         2
3: 3_pp_vox_senado_candidatura          4                     3                 25        0         0                  1      1                     0 

**Lectura de la tabla.** Los números salen de la tabla; los temas de BERTopic se interpretaron a partir de sus palabras clave, sin leer las notas.

- **Los dos métodos coinciden en la mayoría de los temas:** 16 de los 24 temas de BERTopic tienen al menos el 75% de sus notas en un único grupo de KMeans. Por ejemplo, el tema de goles y torneos tiene 250 de sus 270 notas en el grupo de deportes, el de temperaturas y clima tiene 58 de 59 en clima, y el de Israel y Netanyahu tiene las 38 en conflictos internacionales.
- **BERTopic divide los grupos grandes en subtemas:** política en varios temas según el país o el hecho (PP y Vox, Venezuela, el INE de México, el Papa), deportes en fútbol y automovilismo, conflictos en Israel y Ucrania, y clima en temperaturas y sismos.
- **BERTopic aísla la tecnología:** el tema de IA, Apple, YouTube y Google tiene 57 de sus 58 notas en el grupo de salud y vida cotidiana de KMeans, que es el más heterogéneo. BERTopic separa un tema que KMeans mantiene mezclado.
- **Ocho temas se reparten entre varios grupos** (menos del 75% en un único grupo):
  - El tema de balón, Kane y Messi (76 notas) se divide entre lotería (42) y deportes (33). Hipótesis (no verificada): el grupo de lotería incluye resultados de quinielas de fútbol, ya que entre sus palabras aparecen `balón` y `bayern`.
  - Tres temas mezclan crimen con otros enfoques: el de dominicanos, Haití y salvadoreños (38 en crimen y 32 en política, de 81), el de streamer y homicidio (57 en crimen y 23 en cultura, de 80) y el de Ucrania, drones y Arabia Saudí (80 de 121 en conflictos y 21 en crimen).
  - Cuatro temas (beca e IMSS, malbec e inundación, pico y placa, aforo y feriado) se reparten entre tres o cuatro grupos. Por sus palabras clave parecen temas locales o puntuales que no coinciden con ningún grupo de KMeans (hipótesis no verificada).
- **Las 271 notas sin tema (11%) no se reparten por igual:** son el 21% del grupo de economía, el 19% del de elecciones en Brasil y el 16% del de cultura, contra el 1% en deportes y en lotería. Hipótesis (no verificada): los grupos con vocabulario muy específico (deportes, lotería) forman zonas densas fáciles de detectar, mientras que los más amplios dejan más notas sin asignar.

### 4.5 Conclusión

- Los embeddings permiten encontrar estructura temática sin etiquetas en un corpus de 2500 notas: KMeans da una visión general de 10 grupos y BERTopic una más fina de 24 temas, con un 11% de notas sin asignar.
- Los dos métodos coinciden en los temas más definidos: deportes, clima, cultura, Israel y las elecciones en Brasil tienen casi todas sus notas en un único grupo de KMeans. BERTopic además separa subtemas que KMeans mantiene unidos, como tecnología (IA, Apple, YouTube) dentro del grupo de salud y vida cotidiana.
- Los silhouette de KMeans son bajos (alrededor de 0.10). `k=10` es el máximo, pero `k=8` (0.089) y `k=12` (0.094) son casi equivalentes, así que los grupos no están claramente separados y `k` no es una elección firme.
- En BERTopic hubo que fijar explícitamente `min_cluster_size` y `min_samples`. Con `min_cluster_size=10` salían 82 temas, muchos de ellos un único hecho publicado en varias notas, y con `min_cluster_size=30` y `min_samples=5` un solo tema reunía 514 notas. Se eligió `min_cluster_size=30` y `min_samples=3`, que da 24 temas de tamaño parejo. En una etapa anterior, con un corpus más chico (683 notas), el valor por defecto de `min_samples` también produjo solo 4 temas, con uno que reunía el 78% de las notas.

**Limitaciones:** el corpus cubre aproximadamente 3 días, por lo que algunos grupos están dominados por un único hecho publicado en varias notas, como las elecciones en Brasil. Los dos métodos usan los mismos embeddings, así que su coincidencia no es una validación independiente. Los parámetros se eligieron mirando los resultados y los nombres de los grupos de KMeans se asignaron a mano. La interpretación se basó en pocos títulos por grupo (KMeans) y en las palabras clave de cada tema (BERTopic), así que es una impresión y no una medición.

## 5. Clasificar por tema sin entrenar

**Tarea:** asignar cada nota a un tema (política, economía, deportes, crimen, cultura, salud,
tecnología) sin entrenar ningún modelo con ejemplos etiquetados.

**Etiquetas para evaluar:** algunas URLs de Infobae incluyen una sección temática. Se usan las
notas de esas secciones como etiqueta de referencia, con este mapeo:

| Sección de la URL | Tema |
|---|---|
| `politica` | política |
| `economia` | economía |
| `deportes` | deportes |
| `judiciales` | crimen |
| `teleshow` | cultura |
| `salud` | salud |
| `tecno` | tecnología |

`judiciales` se asignó a `crimen` por decisión propia, y lo mismo vale para `teleshow` y `cultura`.
`internacional` no tiene una sección equivalente y queda fuera de la evaluación. Las notas de
otras secciones (ediciones por país, `noticias`, `sociedad`, etc.) no tienen etiqueta temática
y no se evalúan.

**Limitaciones:** las etiquetas provienen del criterio del medio y no de una anotación propia, y
solo se evalúa sobre notas que el medio ya ubicó en una sección temática, que pueden ser más
fáciles de clasificar que el resto del corpus.

In [84]:
MAPA_SECCION = {
    "politica": "política",
    "economia": "economía",
    "deportes": "deportes",
    "judiciales": "crimen",
    "teleshow": "cultura",
    "salud": "salud",
    "tecno": "tecnología",
}
d["tema_url"] = d["seccion"].map(MAPA_SECCION)
dt = d[d["tema_url"].notna()]

print(len(dt), "notas con etiqueta temática en la URL")
print(dt["tema_url"].value_counts())

312 notas con etiqueta temática en la URL
tema_url
deportes      62
tecnología    58
salud         50
cultura       49
política      39
economía      33
crimen        21
Name: count, dtype: int64


### 5.1 Zero-shot con embeddings

Se compara el embedding de cada nota con el embedding de una descripción corta de cada tema, y
se asigna el tema más cercano (similitud coseno). No se entrena nada. Las descripciones se
escribieron antes de ver estos resultados y no se ajustan a estas 312 notas, para que la
evaluación no quede sesgada por el ajuste. El modelo lee solo el comienzo de cada nota
(~128 tokens).

In [85]:
DESCRIPCIONES = {
    "política": "Noticia sobre política, gobierno y elecciones",
    "economía": "Noticia sobre economía, finanzas, precios y negocios",
    "deportes": "Noticia sobre deportes, fútbol y competencias",
    "crimen": "Noticia sobre crimen, violencia, atentados y justicia",
    "cultura": "Noticia sobre espectáculos, cultura, premios y arte",
    "salud": "Noticia sobre salud y medicina",
    "tecnología": "Noticia sobre tecnología, ciencia y medio ambiente",
}
nombres_tema = list(DESCRIPCIONES)
E_desc = modelo_emb.encode(list(DESCRIPCIONES.values()))

sims = util.cos_sim(E[dt.index.to_numpy()], E_desc)
y_true = dt["tema_url"]
y_pred = pd.Series([nombres_tema[i] for i in sims.argmax(dim=1).tolist()], index=dt.index)

print("baseline (siempre la más frecuente):", round(y_true.value_counts(normalize=True).max(), 2))
print("accuracy:", round((y_pred == y_true).mean(), 2))
print("F1 macro:", round(f1_score(y_true, y_pred, average="macro"), 2))
print(classification_report(y_true, y_pred))

etq = sorted(DESCRIPCIONES)
print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=etq),
                   index=[f"real_{e}" for e in etq],
                   columns=[f"pred_{e}" for e in etq]))

baseline (siempre la más frecuente): 0.2
accuracy: 0.67
F1 macro: 0.65
              precision    recall  f1-score   support

      crimen       0.43      0.57      0.49        21
     cultura       0.56      0.47      0.51        49
    deportes       0.84      0.85      0.85        62
    economía       0.65      0.73      0.69        33
    política       0.63      0.92      0.75        39
       salud       0.76      0.74      0.75        50
  tecnología       0.65      0.41      0.51        58

    accuracy                           0.67       312
   macro avg       0.64      0.67      0.65       312
weighted avg       0.67      0.67      0.66       312

                 pred_crimen  pred_cultura  pred_deportes  pred_economía  \
real_crimen               12             0              0              4   
real_cultura               9            23              4              0   
real_deportes              0             3             53              2   
real_economía              1

In [86]:
def ver_errores(real, pred, n=10):
    m = (y_true == real) & (y_pred == pred)
    print(f"--- real={real}, predicho={pred}: {m.sum()} notas ---")
    for t in dt.loc[m, "titulo"].head(n):
        print(" -", t)

ver_errores("tecnología", "cultura")
ver_errores("salud", "tecnología")
ver_errores("tecnología", "política")

--- real=tecnología, predicho=cultura: 15 notas ---
 - El director de Google Argentina habló sobre YouTube: “La gente es la que hoy dicta el prime time en TV”
 - Cómo tener karaoke gratis en casa con inteligencia artificial: el truco para quitar la voz
 - Los dos juegos de terror que Epic Games Store regala hasta el 29 de octubre
 - Los rumores de Half-Life 3 vuelven: el capítulo de la saga de Valve más esperado
 - Las guerreras K-pop en Halloween: cuál es el disfraz más buscado, según Google
 - YouTube en Argentina: los 10 videos que son tendencia hoy
 - YouTube en Chile: la lista de los 10 videos más reproducidos que son tendencia hoy
 - YouTube en Colombia: los 10 videos que son tendencia hoy
 - Xuper TV APK v7.2.6 Descarga Gratis (Oficial) 2026: por qué instalar esta app es la peor decisión
 - Elon Musk prometió una película completa sobre ‘La Odisea’ hecha con IA y mostró apenas siete minutos
--- real=salud, predicho=tecnología: 8 notas ---
 - Por qué comer siempre a la misma hora

**Interpretación.** El método acierta 209 de 312 notas (accuracy 0.67, F1 macro 0.65), contra 0.20
del baseline de predecir siempre `deportes`. Deportes (precisión 0.84, recall 0.85) y salud (0.76 y
0.74) se clasifican mejor. Política tiene recall alto (0.92) pero precisión baja (0.63): atrae notas de otras categorías. Tecnología (recall 0.41) y cultura (0.47) pierden más de la mitad de sus notas.

Los cuatro errores más grandes, leídos en los títulos (no en el texto completo):

- **Tecnología → cultura (15 notas, 26% de tecnología; se leyeron 10):** casi todas tratan de YouTube, videojuegos o contenido generado con IA. La sección de tecnología del medio incluye este contenido y la descripción de `cultura` se superpone con él: es un problema de frontera entre categorías más que un error claro del método.
- **Cultura → crimen (9 notas):** cuatro son el mismo hecho (una denuncia por maltrato), dos tratan amenazas o acoso, y tres no tienen contenido delictivo aparente y parecen errores del método.
- **Salud → tecnología (8 notas):** cinco son estudios o historias de ciencia, y la descripción de `tecnología` incluye "ciencia", lo que explicaría parte de la confusión (hipótesis, no verificada). Dos son consejos de salud sin ciencia y parecen errores del método, y una (decoración) no encaja en ninguna categoría.
- **Tecnología → política (7 notas):** dos tienen contenido político (una donación a las elecciones legislativas y una ley sobre avisos con IA). Las otras cinco no tienen contenido político aparente por el título y parecen errores del método.

**Limitaciones:** las etiquetas provienen de la sección de la URL, es decir, del criterio del medio, y no de una anotación propia. Solo se evalúan las 312 notas ubicadas en una sección temática, que pueden ser más fáciles que el resto del corpus. Un mismo hecho puede aparecer en varias notas, lo que pesa mucho en categorías chicas como `crimen` (21 notas). Las descripciones de los temas no se ajustaron a estas notas; la de `tecnología` incluye "ciencia y medio ambiente", lo que se superpone con salud.

## 6. Conclusiones generales

**Corpus:** 2500 notas del feed RSS de Infobae, publicadas entre el 23 de octubre de 2026 (19:00 UTC) y el 27 de octubre de 2026 (03:27 UTC), es decir, unas 80 horas.

**Resultados principales**

| Tarea | Método | Resultado |
|---|---|---|
| Predecir la edición (5 clases, 1852 notas) | Baseline: siempre `america` | accuracy 0.42 |
| | TF-IDF + regresión logística, nota completa | F1 macro 0.85 |
| | TF-IDF + regresión logística, primeras 90 palabras | F1 macro 0.79 |
| | Embeddings + regresión logística | F1 macro 0.65 |
| Descubrir temas sin etiquetas | KMeans | k=10 (silhouette 0.100) |
| | BERTopic | 24 temas, 271 notas sin tema (11%) |
| Clasificar por tema sin entrenar (7 clases, 312 notas) | Baseline: siempre `deportes` | accuracy 0.20 |
| | Zero-shot con embeddings | accuracy 0.67, F1 macro 0.65 |

**Qué se aprendió**

1. **El método más útil depende de la tarea.** Para predecir la edición, TF-IDF superó a los embeddings por 20 puntos de F1 macro con la nota completa (0.85 contra 0.65) y por 14 con texto de largo comparable (0.79 contra 0.65). Hipótesis (no verificada): la edición se delata por palabras concretas (lugares, políticos, instituciones) que TF-IDF ve directamente, mientras que un embedding resume el tema. La confusión principal fue entre `america` y `espana` (203 de los 313 errores, 65%).
2. **Los embeddings permitieron explorar y clasificar por tema sin entrenar.** KMeans y BERTopic coinciden en la mayoría de los temas (16 de 24 temas de BERTopic tienen al menos el 75% de sus notas en un único grupo de KMeans), y el zero-shot acertó el 67% de las notas contra el 20% del baseline. En estas dos tareas no se comparó contra TF-IDF.
3. **Los controles y los valores por defecto importan.** Recortar las notas a 90 palabras separó cuánto de la ventaja de TF-IDF venía del largo del texto (6 puntos) y cuánto del método (14 puntos). En BERTopic, `min_cluster_size=10` produjo 82 temas, muchos de ellos un único hecho publicado en varias notas. Con `min_cluster_size=30` y `min_samples=5` un solo tema reunió 514 notas, y en una etapa anterior, con 683 notas, el valor por defecto de `min_samples` dio solo 4 temas.
4. **Leer los errores cambia su interpretación.** En el zero-shot, parte de los errores no son fallas claras del método: tecnología → cultura (15 notas) mezcla YouTube, videojuegos y contenido con IA, que están en la frontera entre ambas categorías, y cuatro de las nueve notas de cultura → crimen son un mismo hecho (una denuncia por maltrato).

**Limitaciones**

- El corpus cubre unas 80 horas. Varios temas corresponden a un único hecho publicado en varias notas (un grupo de KMeans de 193 notas trata casi solo del balotaje de Brasil), y esas notas pueden quedar repartidas entre entrenamiento y prueba en la validación cruzada, lo que puede inflar los resultados.
- El feed solo entrega las últimas notas, por lo que los períodos en que el script no se ejecutó quedan sin cubrir.
- Los hiperparámetros (`C`, `class_weight`, `k`, `min_cluster_size`, `min_samples`) se eligieron mirando resultados del mismo corpus, así que los números pueden ser algo optimistas.
- Las etiquetas de la sección 5 provienen de la sección de la URL (criterio del medio, no anotación propia), cubren solo 312 notas y `crimen` tiene apenas 21.
- KMeans, BERTopic y el zero-shot usan los mismos embeddings, que leen solo los primeros ~128 tokens de cada nota. Su coincidencia no es una validación independiente.
- La interpretación de los grupos se basó en pocos títulos por grupo y en palabras clave: es una lectura cualitativa, no una medición.

